In [ ]:
# Cell 0 — Install dependencies (Unsloth canonical Colab branch) + clone feat/corpo
#
# Mirrors the install logic from Unsloth's Qwen2.5_(3B)-GRPO notebook, cells 4+5:
#   https://raw.githubusercontent.com/unslothai/notebooks/main/nb/Qwen2.5_(3B)-GRPO.ipynb
#
# Why this exact shape (do not "simplify"):
#   - UNSLOTH_VLLM_STANDBY=1   → +30% context-length headroom (unsloth-specific)
#   - upgrade `uv` first       → uv's resolver is stricter than pip's; avoids the
#                                pip-picks-wrong-trl mistakes we hit on Path B
#   - GPU-aware vllm/triton    → T4 needs vllm==0.9.2 + triton==3.2.0; A100/L4/H100
#                                use vllm==0.15.1 + latest triton. Latest vllm
#                                doesn't work on T4 (silent crash on import).
#   - ONE uv-call bundle       → vllm + numpy + pil + torchvision + bitsandbytes +
#                                xformers + unsloth resolved together so versions
#                                stay consistent (no pip-installs-X-then-uv-overrides-Y).
#   - trl==0.22.2 --no-deps    → avoids TRL 0.24's vllm_ascend + mergekit imports.
#                                --no-deps so trl doesn't drag in transitive packages
#                                that would fight the unsloth-bundled versions.
#   - transformers==4.56.2     → matches what unsloth's wheel was built against.
#   - peft==0.17.1 --no-deps   → 0.18+ added _maybe_shard_state_dict_for_tp which
#                                imports transformers.integrations.tensor_parallel.
#                                EmbeddingParallel — that symbol doesn't exist
#                                until transformers 4.57+. Pinning to 0.17.1 (last
#                                pre-TP release, 2025-08-21) avoids the conflict.
import os
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

!pip install --upgrade -qqq uv

if "COLAB_" not in "".join(os.environ.keys()):
    # Non-Colab environment (local dev, RunPod, etc.): unsloth's simple path
    !pip install unsloth vllm
else:
    # Resolve currently-installed numpy + pillow versions to avoid churning them
    try:
        import numpy, PIL
        _numpy = f"numpy=={numpy.__version__}"
        _pil   = f"pillow=={PIL.__version__}"
    except Exception:
        _numpy, _pil = "numpy", "pillow"

    # GPU-aware vllm + triton pinning
    try:
        import subprocess
        is_t4 = "Tesla T4" in str(subprocess.check_output(["nvidia-smi"]))
    except Exception:
        is_t4 = False
    _vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")

    !uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
    !uv pip install -qqq {_triton}
    !uv pip install -qqq --no-deps --upgrade "torchao>=0.16.0"

!uv pip install -qqq transformers==4.56.2
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip install -qqq --no-deps peft==0.17.1

# Project-specific extras (not part of the canonical Unsloth recipe):
#   openai — DeepSeek V4-Pro judge HTTP client
!uv pip install -qqq "openai>=1.0.0"

# Clone the project at feat/corpo (private repo)
from google.colab import userdata
GITHUB_PAT = userdata.get('GITHUB_PAT')
!rm -rf /content/sft
!git clone --branch feat/corpo --depth 1 \
    https://{GITHUB_PAT}@github.com/deepanathanrajendiran-hub/sft-code-review.git \
    /content/sft
!cp /content/sft/*.py /content/sft/pyproject.toml /content/
!cp -r /content/sft/tests /content/
os.chdir("/content")

!ls /content/corpo_*.py /content/swecare_*.py /content/ood_metrics.py
print("\nVersion check:")
import trl, vllm, torch, datasets, peft, transformers
print(f"  trl          : {trl.__version__}     (expected 0.22.2)")
print(f"  transformers : {transformers.__version__}    (expected 4.56.2)")
print(f"  vllm         : {vllm.__version__}     (T4=0.9.2, else=0.15.1)")
print(f"  torch        : {torch.__version__}")
print(f"  datasets     : {datasets.__version__}")
print(f"  peft         : {peft.__version__}    (expected 0.17.1)")

In [ ]:
# Cell 1 — Mount Drive, load secrets, verify v4 backup
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
os.environ['DEEPSEEK_API_KEY'] = userdata.get('DEEPSEEK_API_KEY')
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'us-west-2'

# v4 adapter paths — USER MUST ensure backup exists before this cell runs
V4_ADAPTER = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces'
V4_BACKUP  = '/content/drive/MyDrive/sft/code-reviewer-lora-v4-traces-backup'
assert os.path.exists(V4_BACKUP + '/adapter_config.json'), \
    f"v4 backup missing! Create one BEFORE running: cp -r {V4_ADAPTER} {V4_BACKUP}"
print(f"v4 adapter:  {V4_ADAPTER}")
print(f"v4 backup:   {V4_BACKUP}")

In [ ]:
# Cell 2 — Build train/eval splits, extract CLEAN defect labels, and score v4 (THE GATE)
# v5: no base-sample cache / no opponent. We extract clean, grounded defect tuples from the
# human PR threads (label_defects.py) and measure v4's JUDGE-INDEPENDENT recall + hallucination.
# Requires DEEPSEEK_API_KEY (Cell 1). No GPU needed for this cell.
import json, os, random

# --- training prompts from dev split; eval from test split (disjoint) ---
!python /content/swecare_loader.py --split dev \
    --output /content/ood_dev_prompts_raw.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl
with open('/content/ood_dev_prompts_raw.jsonl') as f:
    dev_rows = [json.loads(l) for l in f if l.strip()]
sample = random.Random(42).sample(dev_rows, min(1500, len(dev_rows)))
with open('/content/ood_train_prompts.jsonl', 'w') as f:
    for r in sample: f.write(json.dumps(r) + '\n')
!python /content/swecare_loader.py --split test \
    --output /content/ood_input.jsonl \
    --train-jsonl /content/drive/MyDrive/sft/train_dataset_clean.jsonl
print(f"dev pool {len(dev_rows)}   train sample {len(sample)}")

# --- Stage 1: extract clean defect tuples (drops questions/replies/style nits; grounds to diff) ---
os.makedirs('/content/cache', exist_ok=True)
!python /content/label_defects.py --input /content/ood_train_prompts.jsonl --output /content/cache/defect_labels_train.jsonl
!python /content/label_defects.py --input /content/ood_input.jsonl        --output /content/cache/defect_labels_eval.jsonl

# --- THE GATE: v4 (and base) recall + hallucination on the clean EVAL labels ---
# Uses the existing ood_preds_v4.jsonl (already has v4_pred + base_pred).
!python /content/score_v5.py \
    --preds /content/drive/MyDrive/sft/ood_preds_v4.jsonl \
    --labels /content/cache/defect_labels_eval.jsonl \
    --pred-fields v4_pred base_pred

print("\n[GATE] Decision:")
print("  - v4 recall well below 1.0  -> headroom exists, RL is viable. Record v4 recall + halluc; proceed to Cell 3.")
print("  - v4 recall already ~saturated -> RL can only restore, not exceed. STOP and pivot to data (v4.1).")
print("  - This v4 recall/halluc pair IS the bar Cell 5/Cell 7 must beat (recall UP, halluc <= v4).")

In [ ]:
# Cell 3 — Pre-training variance gate + auto-pick R_min for v5 (~10-15 min, ~$1 DeepSeek)
#
# v5 scores v4 rollouts with the VERIFIABLE reward (F1 on labeled + claim-penalty on clean +
# grounding + length) — no opponent, no quality judge. The gate confirms the reward has
# within-group spread (>=0.10) so advantages don't collapse, and prints p25/p33/p40/p50
# R_min candidates (CoRPO correctness boundary). This cell auto-extracts the p33 default.
import subprocess, re, sys

print("[cell3] running v5 variance gate (verifiable reward against clean defect labels)...")
result = subprocess.run(
    ["python", "/content/corpo_train.py", "--variance-gate-only",
     "--v4-adapter", V4_ADAPTER,
     "--v4-backup",  V4_BACKUP,
     "--train-prompts", "/content/ood_train_prompts.jsonl",
     "--defect-labels", "/content/cache/defect_labels_train.jsonl",
     "--output-dir", "/content/corpo-out"],
    capture_output=True, text=True,
)
print(result.stdout)
print(result.stderr, file=sys.stderr)

# Distinguish a real gate FAIL (flat reward) from a crash (code/env error). The gate only
# prints "verdict: PASS/FAIL" if it ran to completion; a traceback means it crashed.
reached_verdict = ("[variance-gate] verdict:" in result.stderr)
if result.returncode != 0:
    if not reached_verdict:
        raise RuntimeError(
            "[variance-gate] CRASHED before a verdict (see traceback above) — a code/env error, "
            "NOT a flat-reward failure. Fix the error (e.g. git pull + re-copy /content/*.py) and re-run."
        )
    raise RuntimeError(
        "[variance-gate] FAIL — within-group reward std < 0.10 (flat reward). Do NOT train. "
        "Inspect the histogram above; the fix is to rebalance the reward, not to train. Paste the output."
    )

m = re.search(r"recommended default: p33 = ([\d.]+)", result.stderr)
if not m:
    raise RuntimeError("[variance-gate] PASSED but couldn't parse R_min; set R_MIN manually from the printout.")
R_MIN = float(m.group(1))
print(f"\n[cell3] PASS — auto-selected R_MIN = {R_MIN}  (p33 of the v4 verifiable-reward distribution)")
print(f"[cell3] If the histogram looks bimodal, set R_MIN at the trough manually, then run Cell 4.")

In [ ]:
# Cell 4 — Train v5 (verifiable-reward CoRPO) — fixes every Run #1-3 defect
#
# Why v5 (after Runs #1/#2/#3 all regressed: -22pp, -52pp, -6pp):
#   The judge-reward path was structurally broken — wrong opponent (vs BASE, not v4),
#   beta=0 drift, and a length/structure-gameable judge that saw only review[:500].
#   v5 replaces it with a VERIFIABLE reward (no opponent, no judge):
#     R = 0.6*recall(clean defect tuples, semantic matcher)   # find real bugs
#       + 0.3*grounding(1 - halluc vs diff)                   # ANTI-HALLUCINATION (goal half 2)
#       + 0.1*length_sanity
#   Goodhart can't apply (the reward IS the eval metric); verbosity earns nothing because
#   the matcher requires actually identifying the defect, not name-dropping it.
#
# v5 knobs vs Run #3:
#   --defect-labels  : clean grounded labels (replaces --base-cache / opponent)
#   --kl-beta 0.02   : RESTORE the v4 anchor (Run #3 beta=0 caused capability/format drift)
#   --r-min-correct  : p33 of the v4 verifiable-reward distribution (Cell 3)
#   loss_type=dr_grpo, scale_rewards=none : kept (correct Dr.GRPO combo)

!python /content/corpo_train.py \
    --v4-adapter {V4_ADAPTER} \
    --v4-backup {V4_BACKUP} \
    --train-prompts /content/ood_train_prompts.jsonl \
    --defect-labels /content/cache/defect_labels_train.jsonl \
    --output-dir /content/corpo-out \
    --checkpoint-sync-dir /content/drive/MyDrive/sft/corpo-out-v5 \
    --r-min-correct {R_MIN} \
    --kl-beta 0.02 \
    --learning-rate 5e-6 \
    --num-generations 8 \
    --prompts-per-step 4 \
    --max-new-tokens 2048 \
    --epochs 1 \
    --checkpoint-every 75 \
    --copy-to /content/drive/MyDrive/sft/code-reviewer-lora-v5-verifiable

# RESUME after a Colab disconnect: the local /content/corpo-out is wiped, but checkpoints
# are mirrored to Drive. Re-run Cells 0-1, then re-run THIS cell with --resume added,
# pointing at the latest Drive checkpoint, e.g.:
#   --resume /content/drive/MyDrive/sft/corpo-out-v5/checkpoint-150
# (batch note: per_device_train_batch_size=num_generations=8, gradient_accumulation_steps=
#  prompts_per_step=4 -> 4 prompts/optimizer-step; 8 % 8 == 0 satisfies TRL's divisibility rule.)

In [ ]:
# Cell 5 — mid-eval all v5 checkpoints vs the v4 bar (defect_recall / fp_rate / halluc)
#
# Runs as a SUBPROCESS: in-cell vLLM fails in Colab/Jupyter (vLLM v1 calls sys.stdout.fileno(),
# which a notebook stdout doesn't support). mid_eval.py LoRA-swaps each checkpoint over the base
# (one vLLM load), generates the fixed 50-prompt subset at deployment settings, and scores with
# the precision-aware judge-independent score_v5. Prints, per checkpoint, deltas vs the v4 bar.
!cd /content/sft && git pull --ff-only && cp /content/sft/*.py /content/
!python /content/mid_eval.py \
    --checkpoint-root /content/drive/MyDrive/sft/corpo-out-v5 \
    --v4-preds /content/drive/MyDrive/sft/ood_preds_v4.jsonl \
    --labels /content/cache/defect_labels_eval.jsonl \
    --n-samples 50
# Read the printed table: a checkpoint "beats v4" iff defect_recall > v4 AND fp_rate <= v4 AND halluc <= v4.
# Set BEST_CHECKPOINT to the winner (or /content/corpo-out/final after a full run) for Cell 6.

In [ ]:
# Cell 6 — Verify chat_template parity, merge BEST_CHECKPOINT, generate v5 preds on full 632 OOD set
#
# Uses BEST_CHECKPOINT from Cell 5. Falls back to /content/corpo-out/final if Cell 5 was skipped.

# 6a. chat_template parity (else run_ood_eval.py's assert fires mid-run)
from transformers import AutoTokenizer
v4_tok = AutoTokenizer.from_pretrained(V4_ADAPTER)
base_tok = AutoTokenizer.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct')
assert v4_tok.chat_template == base_tok.chat_template, \
    "v4 chat_template differs from base — patch run_ood_eval.py to load tokenizer per-model"
del v4_tok, base_tok
print("[cell6] chat_template parity: OK")

# 6b. which checkpoint
try:
    _src = BEST_CHECKPOINT
    print(f"[cell6] using BEST_CHECKPOINT from Cell 5: {_src}")
except NameError:
    _src = '/content/corpo-out/final'
    print(f"[cell6] Cell 5 skipped — falling back to {_src}")

# 6c. merge v5 adapter (vLLM eval needs a full model)
import gc, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained('unsloth/Qwen2.5-Coder-7B-Instruct', dtype=torch.bfloat16)
merged = PeftModel.from_pretrained(base, _src).merge_and_unload()
merged.save_pretrained('/content/sft-v5-merged-for-eval', safe_serialization=True)
AutoTokenizer.from_pretrained(V4_ADAPTER).save_pretrained('/content/sft-v5-merged-for-eval')
del merged, base
gc.collect(); torch.cuda.empty_cache()

# 6d. generate v5 predictions on the 632 OOD set (--skip-base: base preds already in ood_preds_v4.jsonl)
!python /content/run_ood_eval.py \
    --input /content/ood_input.jsonl \
    --output /content/ood_preds_v5.jsonl \
    --v4-model /content/sft-v5-merged-for-eval \
    --skip-base

In [ ]:
# Cell 7 — v5 vs v4 on the FULL 632 OOD set (judge-independent): defect_recall + fp_rate + halluc + verdict
#
# Goal is binary, measured WITHOUT any quality judge:
#   SHIP v5  iff  defect_recall(v5) > defect_recall(v4)  AND  fp_rate(v5) <= fp_rate(v4)  AND  halluc(v5) <= halluc(v4)
import json, sys
sys.path.insert(0, '/content')
import score_v5

labels = {}
for l in open('/content/cache/defect_labels_eval.jsonl'):
    if l.strip():
        r = json.loads(l); labels[r['instance_id']] = r.get('defects', [])

def _load(p):
    return [json.loads(l) for l in open(p) if l.strip()]

# Both models' outputs are stored under 'v4_pred' (run_ood_eval writes the loaded model's output there).
print("scoring v4 (632, parallel)..."); v4 = score_v5.score(_load('/content/drive/MyDrive/sft/ood_preds_v4.jsonl'), labels, 'v4_pred')
print("scoring v5 (632, parallel)..."); v5 = score_v5.score(_load('/content/ood_preds_v5.jsonl'), labels, 'v4_pred')

def _dr(s): return s['defect_recall_labeled'] if s['defect_recall_labeled'] is not None else 0.0
def _fp(s): return s['fp_rate_clean'] if s['fp_rate_clean'] is not None else 1.0
def _pr(s): return s['precision_labeled'] if s['precision_labeled'] is not None else 0.0

print(f"\n{'metric':16s} {'v4':>8} {'v5':>8} {'delta':>9}")
for name, fn in [('defect_recall', _dr), ('precision', _pr),
                 ('fp_rate(clean)', _fp), ('halluc', lambda s: s['halluc_mean'])]:
    a, b = fn(v4), fn(v5)
    print(f"{name:16s} {a:>8.3f} {b:>8.3f} {b-a:>+9.3f}")

ship = (_dr(v5) > _dr(v4)) and (_fp(v5) <= _fp(v4) + 1e-9) and (v5['halluc_mean'] <= v4['halluc_mean'] + 1e-9)
print(f"\nVERDICT: {'SHIP v5 — defect_recall UP and (fp_rate, halluc) NOT worse than v4' if ship else 'KEEP v4 — v5 did not clear all three (recall up, fp_rate<=v4, halluc<=v4)'}")
print("(n=632, judge-independent. defect_recall is the primary lift; fp_rate/halluc are the 'less hallucination' guards.)")
json.dump({'v4': v4, 'v5': v5, 'ship': ship}, open('/content/v5_final_verdict.json', 'w'), indent=2)